The csv files are generated by the [VAE_3.0_spike_statistics.ipynb]


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import re

# 0: The Statistics
We will use the statistics as a user

In [6]:
mean = 0.16
median = 0.2
std = 0.1
min = 0.02
max = 4
gradient = 0.18
statistics_df = pd.DataFrame({'mean': [mean], 'median': [median], 'std': [std], 'min': [min], 'max': [max], 'gradient': [gradient]})
X = [mean, median, std, min, max, gradient]
X = statistics_df.values
kmoid_cluster = joblib.load('../Data_preprocess/kmedoids_model.joblib')
cluster = kmoid_cluster.predict(X)[0]
statistics_df['kmedoid_clusters'] = cluster
print(statistics_df)

   mean  median  std   min  max  gradient  kmedoid_clusters
0  0.16     0.2  0.1  0.02    4      0.18                 6


# 1. Number of spikes for a day

In [7]:
def generate_random_spike_count(cluster, month):
    spike_num_prob_df = pd.read_csv('../../Data_processed/Spike_count_cluster_month.csv')
    temp_df = spike_num_prob_df[spike_num_prob_df['kmedoid_clusters'] == cluster]
    selected_spike_count = np.random.choice(temp_df['spike_count'], p=temp_df[f'month_{month}'])
    return selected_spike_count
    

spike_count = generate_random_spike_count(0, 2)
print(spike_count)

4.0


In [12]:
spike_num_prob_df = pd.read_csv('../../Data_processed/Spike_count_cluster_month.csv')
spike_num_prob_df

,kmedoid_clusters,spike_count,month_1,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,0,1,0.403600,0.409409,0.425867,0.487109,0.521268,0.551985,0.609058,0.603919,0.544037,0.510842,0.449420,0.423755
1,0,2,0.320185,0.321335,0.308708,0.304457,0.306922,0.297936,0.265801,0.270102,0.287942,0.296274,0.310558,0.309031
2,0,3,0.186209,0.181510,0.182237,0.148619,0.126415,0.110893,0.097207,0.094194,0.120183,0.138133,0.162521,0.178106
3,0,4,0.090006,0.087746,0.083188,0.059814,0.045395,0.039185,0.027935,0.031785,0.047837,0.054750,0.077501,0.089108
4,1,1,0.442430,0.464061,0.482127,0.548642,0.582584,0.605536,0.644160,0.629086,0.593736,0.557135,0.500151,0.470179
5,1,2,0.327955,0.317196,0.313598,0.295190,0.283810,0.276905,0.255877,0.262843,0.283833,0.292666,0.318702,0.312810
6,1,3,0.162948,0.154934,0.146110,0.118386,0.102371,0.090941,0.076641,0.081543,0.092624,0.111771,0.130026,0.153442
7,1,4,0.066667,0.063809,0.058166,0.037781,0.031235,0.026617,0.023321,0.026527,0.029807,0.038428,0.051122,0.063570
8,2,1,0.449123,0.451854,0.473188,0.526164,0.567959,0.595081,0.636706,0.629845,0.574485,0.548736,0.480089,0.456102
9,2,2,0.312523,0.311150,0.301872,0.287234,0.288128,0.265104,0.258163,0.249471,0.278905,0.280578,0.299709,0.305575


# 2. When the spike(s) occur

In [5]:
# Function to manually parse and format the time_quad string
def parse_and_format_time(time_quad_str):
    # Extract all time occurrences using regex
    times = re.findall(r'\d+, \d+', time_quad_str)
    # Convert each found time into a formatted string
    formatted_times = [f"{int(hour):02d}:{int(minute):02d}:00" for hour, minute in (time.split(', ') for time in times)]
    return tuple(formatted_times)

In [6]:
def get_spike_time(cluster, spike_count):
    spike_time_prob_df = pd.read_csv(f'../../Data_processed/statistics/{spike_count}spike_time_pair.csv')
    if spike_count == 1:
        pass
    else:
        spike_time_prob_df['time_pairs'] = spike_time_prob_df['time_pairs'].apply(parse_and_format_time)

    spike_time_prob_df = spike_time_prob_df[spike_time_prob_df['clusters'] == cluster]
    
    spike_time_prob_df['probability'] /= spike_time_prob_df['probability'].sum()

    selected_time = np.random.choice(spike_time_prob_df['time_pairs'], p=spike_time_prob_df['probability'])

    return selected_time

spike_time = get_spike_time(cluster, spike_count)
print(spike_time)

('13:30:00', '15:00:00')


In [14]:
spike_time_prob_df = pd.read_csv(f'../../Data_processed/statistics/{2}spike_time_pair.csv')
spike_time_prob_df

,time_pairs,probability,clusters
0,"(datetime.time(0, 0), datetime.time(0, 30))",0.000076,6
1,"(datetime.time(0, 0), datetime.time(1, 0))",0.000405,6
2,"(datetime.time(0, 0), datetime.time(1, 30))",0.000380,6
3,"(datetime.time(0, 0), datetime.time(2, 0))",0.000076,6
4,"(datetime.time(0, 0), datetime.time(2, 30))",0.000076,6
...,...,...,...
10199,"(datetime.time(22, 0), datetime.time(23, 0))",0.001170,0
10200,"(datetime.time(22, 0), datetime.time(23, 30))",0.001040,0
10201,"(datetime.time(22, 30), datetime.time(23, 0))",0.000032,0
10202,"(datetime.time(22, 30), datetime.time(23, 30))",0.001105,0


# 3. What types are the spikes?

In [7]:
def get_spike_type(cluster, spike_count):
    spike_type_prob_df = pd.read_csv(f'../../Data_processed/statistics/{spike_count}spike_type_prob.csv')
    spike_type_prob_df = spike_type_prob_df[spike_type_prob_df['clusters'] == cluster]
    spike_type_prob_df['probability'] /= spike_type_prob_df['probability'].sum()

    selected_spike_type = []
    for i in range(spike_count):
        selected_spike_type.append(np.random.choice(spike_type_prob_df['spike_type'], p=spike_type_prob_df['probability']))
    
    return selected_spike_type


spike_type = get_spike_type(cluster, spike_count)
print(spike_type)

[2, 2]


In [15]:
spike_type_prob_df = pd.read_csv(f'../../Data_processed/statistics/{1}spike_type_prob.csv')
spike_type_prob_df

,clusters,spike_type,probability
0,6,2,0.500818
1,6,3,0.499182
2,9,2,0.548102
3,9,3,0.451898
4,2,2,0.356547
5,2,3,0.643453
6,5,2,0.463800
7,5,3,0.536200
8,4,2,0.365910
9,4,3,0.634090


# 3. Continued, how much pre-spikes and post-spikes for the spike?

In [8]:
def get_pre_post_spike(cluster, spike_count):
    cluster_df_12 = pd.read_csv(f'../../Data_processed/statistics/{spike_count}spike_12_clusters.csv')
    cluster_df_13 = pd.read_csv(f'../../Data_processed/statistics/{spike_count}spike_13_clusters.csv')
    cluster_df_24 = pd.read_csv(f'../../Data_processed/statistics/{spike_count}spike_24_clusters.csv')
    cluster_df_34 = pd.read_csv(f'../../Data_processed/statistics/{spike_count}spike_34_clusters.csv')
    cluster_df_12 = cluster_df_12[cluster_df_12['cluster'] == cluster]
    cluster_df_13 = cluster_df_13[cluster_df_13['cluster'] == cluster]
    cluster_df_24 = cluster_df_24[cluster_df_24['cluster'] == cluster]
    cluster_df_34 = cluster_df_34[cluster_df_34['cluster'] == cluster]
    spike_12 = np.random.choice(cluster_df_12['count_1s_before_2'], p=cluster_df_12['probability'])
    spike_13 = np.random.choice(cluster_df_13['count_1s_before_3'], p=cluster_df_13['probability'])
    spike_24 = np.random.choice(cluster_df_24['count_4s_after_2'], p=cluster_df_24['probability'])
    spike_34 = np.random.choice(cluster_df_34['count_4s_after_3'], p=cluster_df_34['probability'])
    return spike_12, spike_13, spike_24, spike_34

prespike_2, prespike_3, postspike_2, postspike_3 = get_pre_post_spike(cluster, spike_count)
print(prespike_2, prespike_3, postspike_2, postspike_3)

2 2 2 1


In [16]:
cluster_df_12 = pd.read_csv(f'../../Data_processed/statistics/{spike_count}spike_12_clusters.csv')
cluster_df_12

,count_1s_before_2,probability,cluster
0,1,0.602467,6
1,2,0.215901,6
2,3,0.074023,6
3,4,0.047293,6
4,5,0.033585,6
...,...,...,...
65,3,0.083650,0
66,4,0.043515,0
67,5,0.026193,0
68,6,0.015716,0


# 4. The magnitude for the spikes

In [9]:
def get_spike_magnitude(spike_type, cluster, mean, std, max):
    count = len(spike_type)
    spike_mag_stats_df = pd.read_csv('../../Data_processed/statistics/spike_mag_stats.csv')
    spike_mag_stats_df = spike_mag_stats_df[spike_mag_stats_df['clusters'] == cluster]

    spike_mag = []
    for type in spike_type:

        if type == 2:
            min_mag = mean + 2*std
            max_mag = mean + 3*std
        else:
            min_mag = mean + 3*std
            max_mag = max
        
        temp_df = spike_mag_stats_df[spike_mag_stats_df['spike_type'] == type]
        mag = np.random.normal(temp_df['mean'], temp_df['std'])
        while mag < min_mag or mag > max_mag:
            mag = np.random.normal(temp_df['mean'], temp_df['std'])
        spike_mag.append(mag[0])
    return spike_mag

spike_mag = get_spike_magnitude(spike_type, cluster, mean, std, max)
print(spike_mag)
        

[0.37796821999157215, 0.43952745967647555]


In [17]:
spike_mag_stats_df = pd.read_csv('../../Data_processed/statistics/spike_mag_stats.csv')
spike_mag_stats_df

,clusters,spike_type,mean,std
0,6,2,3.508792,0.680900
1,6,3,5.467586,1.945298
2,9,2,3.401628,0.830055
3,9,3,5.566434,2.626187
4,2,2,3.811648,0.738265
5,2,3,6.635555,2.592647
6,5,2,3.409268,0.663323
7,5,3,5.441601,1.976626
8,4,2,3.440811,0.749854
9,4,3,6.109367,2.953351


# All Together

In [10]:
def get_spike_df(cluster, month, mean, std, max):
    spike_df = pd.DataFrame()
    spike_df['time'] = pd.date_range('00:00:00', '23:30:00', freq='30T')
    spike_df['time'] = spike_df['time'].dt.time
    spike_df['spike_type'] = 0
    spike_df['spike_magnitude'] = 1

    spike_count = generate_random_spike_count(cluster, month)
    spike_time = get_spike_time(cluster, spike_count)
    # Turn the spike time into a list
    if spike_count == 1:
        spike_time = [spike_time]
    else:
        spike_time = list(spike_time)
    spike_time = [pd.to_datetime(time).time() for time in spike_time]
    spike_type = get_spike_type(cluster, spike_count)
    spike_mag = get_spike_magnitude(spike_type, cluster, mean, std, max)

    prespike_2, prespike_3, postspike_2, postspike_3 = get_pre_post_spike(cluster, spike_count)

    for i in range(spike_count):
        spike_df.loc[spike_df['time'].isin(spike_time), 'spike_type'] = spike_type[i]
        spike_df.loc[spike_df['time'] == spike_time[i], 'spike_magnitude'] = spike_mag[i]

    for i, time in enumerate(spike_time):
        spike_index = spike_df.index[spike_df['time'] == time].tolist()[0]  # Get the index of the current spike time
        
        # Pre-Spike Logic
        pre_spike_marker = 1 if spike_type[i] == 2 else (1 if spike_type[i] == 3 else None)
        pre_spike_count = prespike_2 if spike_type[i] == 2 else (prespike_3 if spike_type[i] == 3 else 0)
        
        # Post-Spike Logic
        post_spike_marker = 4 if spike_type[i] == 2 else (4 if spike_type[i] == 3 else None)
        post_spike_count = postspike_2 if spike_type[i] == 2 else (postspike_3 if spike_type[i] == 3 else 0)
        
        # Update DataFrame for Pre-Spike
        for j in range(1, pre_spike_count + 1):
            if spike_index - j >= 0 and spike_df.at[spike_index - j, 'spike_type'] == 0:  # Ensure index does not go negative
                spike_df.at[spike_index - j, 'spike_type'] = pre_spike_marker
        
        # Update DataFrame for Post-Spike
        for j in range(1, post_spike_count + 1):
            if spike_index + j < len(spike_df) and spike_df.at[spike_index + j, 'spike_type'] == 0:  # Ensure index does not exceed DataFrame length
                spike_df.at[spike_index + j, 'spike_type'] = post_spike_marker
    spike_df = pd.DataFrame(spike_df)
    return spike_df

spike_df = get_spike_df(cluster, 2, mean, std, max)
spike_df

,time,spike_type,spike_magnitude
0,00:00:00,0,1.000000
1,00:30:00,0,1.000000
2,01:00:00,0,1.000000
3,01:30:00,0,1.000000
4,02:00:00,0,1.000000
5,02:30:00,0,1.000000
6,03:00:00,0,1.000000
7,03:30:00,0,1.000000
8,04:00:00,0,1.000000
9,04:30:00,0,1.000000


In [11]:
import utils
utils.get_48_spike_data()

,time,spike_type,spike_magnitude
0,00:00,0,1.000000
1,00:30,0,1.000000
2,01:00,1,1.000000
3,01:30,1,1.000000
4,02:00,1,1.000000
5,02:30,2,4.077471
6,03:00,4,1.000000
7,03:30,0,1.000000
8,04:00,0,1.000000
9,04:30,0,1.000000
